###Transformation of Airline Data 

In [0]:
print("Notebook attached successfully!")

Notebook attached successfully!


In [0]:
%pip install azure-storage-file-datalake

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
# Replace these placeholders with your values
storage_account_name = "airlinesdatalake123"
container_name = "airlinedata"  
storage_account_key = "" 



spark.conf.set(
  f"fs.azure.account.key.{storage_account_name}.dfs.core.windows.net",
  storage_account_key
)

In [0]:
df = spark.read.csv(
    f"abfss://{container_name}@{storage_account_name}.dfs.core.windows.net/raw-data/airlines_flights_data.csv",
    header=True,
    inferSchema=True
)

df.show(5)

+-----+--------+-------+-----------+--------------+-----+-------------+----------------+-------+--------+---------+-----+
|index| airline| flight|source_city|departure_time|stops| arrival_time|destination_city|  class|duration|days_left|price|
+-----+--------+-------+-----------+--------------+-----+-------------+----------------+-------+--------+---------+-----+
|    0|SpiceJet|SG-8709|      Delhi|       Evening| zero|        Night|          Mumbai|Economy|    2.17|        1| 5953|
|    1|SpiceJet|SG-8157|      Delhi| Early_Morning| zero|      Morning|          Mumbai|Economy|    2.33|        1| 5953|
|    2| AirAsia| I5-764|      Delhi| Early_Morning| zero|Early_Morning|          Mumbai|Economy|    2.17|        1| 5956|
|    3| Vistara| UK-995|      Delhi|       Morning| zero|    Afternoon|          Mumbai|Economy|    2.25|        1| 5955|
|    4| Vistara| UK-963|      Delhi|       Morning| zero|      Morning|          Mumbai|Economy|    2.33|        1| 5955|
+-----+--------+-------+

Convert numeric columns 

In [0]:
from pyspark.sql.functions import col, when

# Convert duration, price, days_left to proper numeric types
df = df.withColumn("Duration", col("duration").cast("double")) \
       .withColumn("Price", col("price").cast("double")) \
       .withColumn("Days_Left", col("days_left").cast("int"))

Fill missing Categorical values if any 

In [0]:
df = df.fillna({
    "stops": "zero",    # default non-stop
    "class": "Economy"  # default economy
})

Add derived columns 

In [0]:
# Is direct flight?
df = df.withColumn("Is_Direct", when(col("stops") == "zero", 1).otherwise(0))

# Encode Class as number for analytics
df = df.withColumn("Class_Num", when(col("class")=="Business", 1).otherwise(0))

Save processed data to adls 

In [0]:
df.write.mode("overwrite").csv(
    f"abfss://{container_name}@{storage_account_name}.dfs.core.windows.net/final-data/airlines_processed.csv",
    header=True
)

### To push Airline data from ADLS to SQL Database : 

1. Load your processed data : 

In [0]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()

# Load processed data from ADLS final-data folder
df = spark.read.format("csv").option("header", "true").load("abfss://airlinedata@airlinesdatalake123.dfs.core.windows.net/final-data/")
df.show(5)

+-----+--------+-------+-----------+--------------+-----+-------------+----------------+-------+--------+---------+-----+
|index| airline| flight|source_city|departure_time|stops| arrival_time|destination_city|  class|duration|days_left|price|
+-----+--------+-------+-----------+--------------+-----+-------------+----------------+-------+--------+---------+-----+
|    0|SpiceJet|SG-8709|      Delhi|       Evening| zero|        Night|          Mumbai|Economy|    2.17|        1| 5953|
|    1|SpiceJet|SG-8157|      Delhi| Early_Morning| zero|      Morning|          Mumbai|Economy|    2.33|        1| 5953|
|    2| AirAsia| I5-764|      Delhi| Early_Morning| zero|Early_Morning|          Mumbai|Economy|    2.17|        1| 5956|
|    3| Vistara| UK-995|      Delhi|       Morning| zero|    Afternoon|          Mumbai|Economy|    2.25|        1| 5955|
|    4| Vistara| UK-963|      Delhi|       Morning| zero|      Morning|          Mumbai|Economy|    2.33|        1| 5955|
+-----+--------+-------+

2. Write to Azure SQL Database

In [0]:
# JDBC connection details
jdbc_url = " "
table_name = "Airlines_Processed"
user = " "
password = " "

# Write DataFrame to SQL Database
df.write \
  .format("jdbc") \
  .mode("overwrite") \
  .option("url", jdbc_url) \
  .option("dbtable", table_name) \
  .option("user", user) \
  .option("password", password) \
  .save()